In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from NN_helper_functions import predict, run_model_epochs
from process_data_higgs import preprocess_data

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

In [ ]:
X_train, Y_train, X_test, Y_test = preprocess_data()

X_train_gpu = cp.asarray(X_train)
Y_train_gpu = cp.asarray(Y_train)
X_test_gpu = cp.asarray(X_test)
Y_test_gpu = cp.asarray(Y_test)

In [ ]:
parameters_lr = {}
costs_lr = {}
layers_dims_lr = [X_train.shape[0], 20, 10, 7, 4, 1]
learning_rates_lr = [0.1, 0.075, 0.05, 0.025, 0.01]

In [ ]:
parameters_arc = {}
costs_arc = {}
layers_dims_arc = {
    "previous":          [X_train.shape[0], 20, 10, 7, 4, 1],
    "double":            [X_train.shape[0], 40, 20, 14, 8, 1],
    "extra layer":       [X_train.shape[0], 40, 20, 10, 7, 4, 1],
    "two extra layers":  [X_train.shape[0], 80, 40, 20, 10, 7, 4, 1],
    "wide":              [X_train.shape[0], 128, 64, 1],
    "very wide":         [X_train.shape[0], 256, 128, 1],
    "deep and wide":     [X_train.shape[0], 128, 64, 32, 16, 8, 1],
    "oversized":         [X_train.shape[0], 512, 256, 128, 64, 1],
}

In [ ]:
for spec in layers_dims_arc:
    print(f"Model: {spec}")
    parameters_arc[spec], costs_arc[spec] = run_model_epochs(X_train_gpu, Y_train_gpu, layers_dims_arc[spec], learning_rate=0.025, 
                                                     num_epochs = 400, batch_size=1024, check_convergence=True, convergence_threshold=0.0001, print_cost=True)

In [ ]:
accuracies_arc = {}
for spec in layers_dims_arc:
    print(f"Model Type: {spec}")
    print("Train: ")
    train_mat, train_acc = predict(X_train_gpu, Y_train_gpu, parameters_arc[spec])
    print("Test: ")
    test_mat, test_acc = predict(X_test_gpu, Y_test_gpu, parameters_arc[spec])
    print()
    accuracies_arc[spec] = {"train": train_acc, "test": test_acc}

In [ ]:
specs = list(accuracies_arc.keys())  # preserves insertion order: previous, double, extra layer, ...
train_acc = [accuracies_arc[spec]["train"] for spec in specs]
test_acc = [accuracies_arc[spec]["test"] for spec in specs]

# Convert to plain floats in case any are CuPy scalars
train_acc = [float(a) for a in train_acc]
test_acc = [float(a) for a in test_acc]

x = np.arange(len(specs))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 8))
bars1 = ax.bar(x - width/2, train_acc, width, label='Train accuracy')
bars2 = ax.bar(x + width/2, test_acc, width, label='Test accuracy')

ax.set_xlabel('Architecture')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs test accuracy by architecture')
ax.set_xticks(x)
ax.set_xticklabels(specs, rotation=20, ha='right')
ax.set_ylim(0.6, 0.87)  # wide range to accommodate the "oversized" collapse
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7, rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

for architecture, cost_list in costs_arc.items():
    # Convert CuPy scalars to plain floats if needed
    cost_values = [float(c) for c in cost_list]
    plt.plot(cost_values, label=f"architecture={architecture}")

plt.xlabel("Iterations (x10)")
plt.ylabel("Cost")
plt.title("Cost convergence across learning rates")
plt.legend()
plt.show()